In [20]:
import pandas as pd

# Dataset is already split into train and test sets, so we can load them directly
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

In [21]:
data_dictionary = {
    "row_id": "identifier",
    "credit_limit": "predictor",
    "sex": "predictor / audit attribute",
    "education": "predictor",
    "marital_status": "predictor",
    "age": "predictor",
    "latest_repayment_status": "predictor",
    "maximum_repayment_delay_6m": "predictor",
    "delayed_months_6m": "predictor",
    "average_bill_6m": "predictor",
    "bill_change_sep_to_apr": "predictor",
    "total_payment_6m": "predictor",
    "payment_to_bill_ratio_6m": "predictor",
    "zero_payment_months_6m": "predictor",
    "utilization_sep": "predictor",
    "repayment_assistance_plan": "predictor",
    "payment_difficulty_next_month": "target",
    "predicted_class": "submission"
}

categorical_columns = [
    "sex",
    "education",
    "marital_status",
    "repayment_assistance_plan",
]

In [23]:
predictor_columns = [
    column for column, role in data_dictionary.items() if "predictor" in role
]
target_column = [
    column for column, role in data_dictionary.items() if role == "target"
]

# Train data
X_train = train_data[predictor_columns]
y_train = train_data[target_column]

# Test data
X_test = test_data[predictor_columns]
#y_test = test_data[target_column]

In [24]:
# Transform categorical variables into one-hot encoded features

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(X_train[categorical_columns])
X_test_cat = encoder.transform(X_test[categorical_columns]) # Fit the encoder only on the training data. This avoids leaking information from your test set into the model.

numerical_columns = [
    col for col in predictor_columns
    if col not in categorical_columns
]

X_train_num = X_train[numerical_columns]
X_test_num = X_test[numerical_columns]

In [25]:
# Combine the one-hot encoded categorical features and the numerical features into a single feature matrix for both training and testing

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoder.get_feature_names_out(categorical_columns),
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoder.get_feature_names_out(categorical_columns),
    index=X_test.index
)

X_train_encoded = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_test_encoded = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)

X_train_encoded.head()

,credit_limit,age,latest_repayment_status,maximum_repayment_delay_6m,delayed_months_6m,average_bill_6m,bill_change_sep_to_apr,total_payment_6m,payment_to_bill_ratio_6m,zero_payment_months_6m,...,education_graduate_school,education_high_school,education_other,education_university,education_unknown,marital_status_married,marital_status_other_or_unknown,marital_status_single,repayment_assistance_plan_no,repayment_assistance_plan_yes
0,80000,54,0,0,0,24600,3700,10300,0.07,0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1,20000,30,0,0,0,21100,21000,6700,0.05,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
2,230000,35,0,0,0,1700,3600,5100,0.49,1,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,150000,51,0,0,0,76100,-8800,15400,0.03,0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
4,500000,43,0,0,0,121600,26200,25000,0.03,0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [26]:
from interpret import show
from interpret.data import ClassHistogram

hist = ClassHistogram().explain_data(X_train, y_train, name='Train Data')
show(hist)

<!-- http://127.0.0.1:7001/2008144376720/ -->

In [28]:
from sklearn.tree import DecisionTreeClassifier
tree = ClassificationTree()
tree.fit(X_train_encoded, y_train)

In [29]:
from interpret import show
tree_global = tree.explain_global(name='Tree')
show(tree_global)

<!-- http://127.0.0.1:7001/2008110786480/ -->